# 1. Load Data from all the stock csv files in directory

In [165]:
import datetime
import glob
import numpy as np
import pandas as pd

# Get CSV files list from a folder
csv_files = glob.glob("./TSLA-Stocks/TSLA16*.csv")

# Read each CSV file into DataFrame
# This creates a list of dataframes
df_list = (pd.read_csv(file) for file in csv_files)

# Concatenate all DataFrames
df   = pd.concat(df_list, ignore_index=True)

## 1.1 Drop duplicates & check for missing values

In [ ]:
dups = len(df['date'])-len(df['date'].drop_duplicates())
print("Before: # of duplicates", dups, ' out of ', len(df) , ' or ', round(dups/len(df),5), '%')

In [166]:
# Drop duplicate entries
df.drop_duplicates(subset=['date'], keep='first', inplace=True)

In [167]:
# verify there are no duplicate values
df["date"].is_unique
dups = len(df['date'])-len(df['date'].drop_duplicates())
print("After: # of duplicates", dups, ' out of ', len(df) , ' or ', round(dups/len(df),5), '%')

True

## 1.2 Print out all dates with confirming # of quotes (23400) and non-conforming number of quotes

In [168]:
df['date2_str']= df['date'][::].str.slice(stop=10)
pd_group_cnt = df.groupby(['date2_str'])['date2_str'].count().to_frame()
print("Confirming / correct number of quotes: 23,400")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']==23400] )
pd.set_option('display.max_rows', None)
print("Dates Missing quotes")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']!=23400] )
pd.set_option('display.max_rows', 10)
df = df.drop('date2_str', axis=1)

Confirming / correct number of quotes: 23,400
            date2_str
date2_str            
2022-03-04      23400
2022-03-07      23400
2022-03-08      23400
2022-03-09      23400
2022-03-10      23400
...               ...
2022-10-03      23400
2022-10-04      23400
2022-10-05      23400
2022-10-06      23400
2022-10-07      23400

[151 rows x 1 columns]
Dates Missing quotes
            date2_str
date2_str            
2022-03-03       5406


In [169]:
# Only want to track average to 3 decimal places.  Otherwise, end up with a lot of digits
df['average'] = df['average'].round(decimals = 3)

# 2.0 Describe data

In [170]:
df = df.sort_index(ascending=True)
#df = df.tail(10000)
#df = df.tail(30000)
df.describe()

,open,high,low,close,volume,average,barCount
count,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06,3.538806e+06
mean,2.792378e+02,2.792584e+02,2.792166e+02,2.792375e+02,1.981523e+01,2.792374e+02,4.281384e+00
std,3.920246e+01,3.920186e+01,3.920304e+01,3.920243e+01,5.397667e+01,3.920246e+01,7.603019e+00
min,2.068667e+02,2.068733e+02,2.068567e+02,2.068733e+02,0.000000e+00,2.068680e+02,0.000000e+00
25%,2.444167e+02,2.444300e+02,2.443967e+02,2.444133e+02,1.000000e+00,2.444140e+02,1.000000e+00
50%,2.790100e+02,2.790300e+02,2.789967e+02,2.790100e+02,7.500000e+00,2.790095e+02,2.000000e+00
75%,3.031067e+02,3.031300e+02,3.030867e+02,3.031067e+02,2.331000e+01,3.031070e+02,5.000000e+00
max,3.841867e+02,3.842900e+02,3.840133e+02,3.841867e+02,3.025596e+04,3.842220e+02,3.700000e+02


In [171]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 3538806 entries, 0 to 3612599
Data columns (total 9 columns):
 #   Column     Dtype  
---  ------     -----  
 0   date       object 
 1   open       float64
 2   high       float64
 3   low        float64
 4   close      float64
 5   volume     float64
 6   average    float64
 7   barCount   int64  
 8   date2_str  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 270.0+ MB


# 3.0 Add computed columns

In [172]:
df.set_index('date')
df = df.sort_index()

In [173]:
# need this column to compute average of averages
df['_weighted_vol_avg'] = df['volume'] * df['average']

In [174]:
# lambda functions

#return 1st value in series
def firstValue(rows):
    return rows.iloc[0]

#return last value in series
def lastValue(rows):
    return rows.iloc[-1]

#return arrow indicator for boxed in values;
#   -1 below lower bound
#    0 inside the box
#   +1 above the max value
def arrow(new_amt, old_amt, box):
    if old_amt == np.nan:
        return np.nan
    if new_amt == np.nan:
        return np.nan
    if (new_amt - old_amt) <= (box * -1):
        return '-1'
    if (new_amt - old_amt) >= box:
        return '1'
    else:
        return '0'

#  lambda function to adds up the last 5 values, excluding the very last value
def sum_last_5(rows):
    #print ("[" , rows[-6:-1], rows[-6:-1].sum(), "]")
    return rows[-6:-1].sum()

#  lambda function returns lowest of the last 5 values, excluding the very last value
def min_last_5(rows):
    return rows.iloc[-6:-1].min()

#  lambda function returns higest of  the last 5 values, excluding the very last value
def max_last_5(rows):
    return rows.iloc[-6:-1].max()

# Store/Save 1 second windows for h1, h2, h3, h4, h5

In [175]:
%%time
df['h1s_high_max'] = df['high'].rolling(window=2).agg( {'maxLast': firstValue})
df['h1s_low_min'] = df['low'].rolling(window=2).agg( {'minLast': firstValue})
df['h1s_barCount_sum'] = df['barCount'].rolling(window=2).agg( {'sumLast': firstValue})
df['h1s_volume_sum'] = df['volume'].rolling(window=2).agg( {'sumLast': firstValue})
df['h1s_average_avg'] = df['average'].rolling(window=2).agg( {'sumLast': firstValue})

In [176]:
%%time
df['h2s_high_max'] = df['high'].rolling(window=3).agg( {'maxLast': firstValue})
df['h2s_low_min'] = df['low'].rolling(window=3).agg( {'minLast': firstValue})
df['h2s_barCount_sum'] = df['barCount'].rolling(window=3).agg( {'sumLast': firstValue})
df['h2s_volume_sum'] = df['volume'].rolling(window=3).agg( {'sumLast': firstValue})
df['h2s_average_avg'] = df['average'].rolling(window=3).agg( {'sumLast': firstValue})


In [177]:
%%time
df['h3s_high_max'] = df['high'].rolling(window=4).agg( {'maxLast': firstValue})
df['h3s_low_min'] = df['low'].rolling(window=4).agg( {'minLast': firstValue})
df['h3s_barCount_sum'] = df['barCount'].rolling(window=4).agg( {'sumLast': firstValue})
df['h3s_volume_sum'] = df['volume'].rolling(window=4).agg( {'sumLast': firstValue})
df['h3s_average_avg'] = df['average'].rolling(window=4).agg( {'sumLast': firstValue})


In [178]:
%%time
df['h4s_high_max'] = df['high'].rolling(window=5).agg( {'maxLast': firstValue})
df['h4s_low_min'] = df['low'].rolling(window=5).agg( {'minLast': firstValue})
df['h4s_barCount_sum'] = df['barCount'].rolling(window=5).agg( {'sumLast': firstValue})
df['h4s_volume_sum'] = df['volume'].rolling(window=5).agg( {'sumLast': firstValue})
df['h4s_average_avg'] = df['average'].rolling(window=5).agg( {'sumLast': firstValue})

## Compute 5 second window summary

In [179]:
%%time
df['h5s_high_max'] = df['high'].rolling(window=6).agg( {'maxLast5': max_last_5})
df['h5s_low_min'] = df['low'].rolling(window=6).agg( {'minLast5': min_last_5})
df['h5s_barCount_sum'] = df['barCount'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['h5s_volume_sum'] = df['volume'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['_h5s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=6).agg({'SumLast5': sum_last_5})

df['h5s_average_avg'] = df['_h5s_weighted_vol_avg_sum'] / df['h5s_volume_sum']
df['h5s_average_avg'] = df['h5s_average_avg'].round(decimals = 3)
df.drop(columns=['_h5s_weighted_vol_avg_sum'])
#

,date,open,high,low,close,volume,average,barCount,date2_str,_weighted_vol_avg,...,h4s_high_max,h4s_low_min,h4s_barCount_sum,h4s_volume_sum,h4s_average_avg,h5s_high_max,h5s_low_min,h5s_barCount_sum,h5s_volume_sum,h5s_average_avg
0,2022-03-11 15:29:54,265.1000,265.1000,265.0967,265.1000,0.00,265.100,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-03-11 15:29:55,265.0867,265.1200,265.0867,265.1100,12.00,265.101,4,2022-03-11,3181.21200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-03-11 15:29:56,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-03-11 15:29:57,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-03-11 15:29:58,265.0900,265.0900,265.0533,265.0533,12.36,265.069,4,2022-03-11,3276.25284,...,265.1000,265.0967,0.0,0.00,265.100,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3612595,2022-07-25 12:29:49,270.1367,270.1367,270.1333,270.1367,4.68,270.135,1,2022-07-25,1264.23180,...,270.0967,270.0967,0.0,0.00,270.097,270.2133,270.0833,5.0,51.00,270.152
3612596,2022-07-25 12:29:50,270.1367,270.1367,270.1333,270.1367,0.00,270.137,0,2022-07-25,0.00000,...,270.0967,270.0967,0.0,0.00,270.097,270.2133,270.0833,5.0,52.68,270.154
3612597,2022-07-25 12:29:51,270.1700,270.1700,270.1700,270.1700,3.00,270.170,1,2022-07-25,810.51000,...,270.2133,270.1767,2.0,12.00,270.205,270.2133,270.0833,5.0,52.68,270.154
3612598,2022-07-25 12:29:52,270.2100,270.2100,270.1533,270.1533,30.00,270.164,3,2022-07-25,8104.92000,...,270.1500,270.0833,2.0,36.00,270.139,270.2133,270.0833,6.0,55.68,270.155


## Compute 10 second window summary

In [180]:
%%time
df['h10s_high_max'] = df['high'].rolling(window=11).agg( {'maxLast5': max_last_5})
df['h10s_low_min'] = df['low'].rolling(window=11).agg( {'minLast5': min_last_5})
df['h10s_barCount_sum'] = df['barCount'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['h10s_volume_sum'] = df['volume'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['_h10s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=11).agg({'SumLast5': sum_last_5})

df['h10s_average_avg'] = df['_h10s_weighted_vol_avg_sum'] / df['h10s_volume_sum']
df['h10s_average_avg'] = df['h10s_average_avg'].round(decimals = 3)
df.drop(columns=['_h10s_weighted_vol_avg_sum'])

,date,open,high,low,close,volume,average,barCount,date2_str,_weighted_vol_avg,...,h5s_low_min,h5s_barCount_sum,h5s_volume_sum,_h5s_weighted_vol_avg_sum,h5s_average_avg,h10s_high_max,h10s_low_min,h10s_barCount_sum,h10s_volume_sum,h10s_average_avg
0,2022-03-11 15:29:54,265.1000,265.1000,265.0967,265.1000,0.00,265.100,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-03-11 15:29:55,265.0867,265.1200,265.0867,265.1100,12.00,265.101,4,2022-03-11,3181.21200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-03-11 15:29:56,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-03-11 15:29:57,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-03-11 15:29:58,265.0900,265.0900,265.0533,265.0533,12.36,265.069,4,2022-03-11,3276.25284,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3612595,2022-07-25 12:29:49,270.1367,270.1367,270.1333,270.1367,4.68,270.135,1,2022-07-25,1264.23180,...,270.0833,5.0,51.00,13777.7550,270.152,270.2133,270.0833,5.0,51.00,270.152
3612596,2022-07-25 12:29:50,270.1367,270.1367,270.1333,270.1367,0.00,270.137,0,2022-07-25,0.00000,...,270.0833,5.0,52.68,14231.6958,270.154,270.2133,270.0833,5.0,52.68,270.154
3612597,2022-07-25 12:29:51,270.1700,270.1700,270.1700,270.1700,3.00,270.170,1,2022-07-25,810.51000,...,270.0833,5.0,52.68,14231.6958,270.154,270.2133,270.0833,5.0,52.68,270.154
3612598,2022-07-25 12:29:52,270.2100,270.2100,270.1533,270.1533,30.00,270.164,3,2022-07-25,8104.92000,...,270.0833,6.0,55.68,15042.2058,270.155,270.2133,270.0833,6.0,55.68,270.155


## Compute 15 second window summary

In [181]:
%%time
df['h15s_high_max'] = df['high'].rolling(window=16).agg( {'maxLast5': max_last_5})
df['h15s_low_min'] = df['low'].rolling(window=16).agg( {'minLast5': min_last_5})
df['h15s_barCount_sum'] = df['barCount'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['h15s_volume_sum'] = df['volume'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['_h15s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=16).agg({'SumLast5': sum_last_5})

df['h15s_average_avg'] = df['_h15s_weighted_vol_avg_sum'] / df['h15s_volume_sum']
df['h15s_average_avg'] = df['h15s_average_avg'].round(decimals = 3)
df.drop(columns=['_h15s_weighted_vol_avg_sum'])

,date,open,high,low,close,volume,average,barCount,date2_str,_weighted_vol_avg,...,h10s_low_min,h10s_barCount_sum,h10s_volume_sum,_h10s_weighted_vol_avg_sum,h10s_average_avg,h15s_high_max,h15s_low_min,h15s_barCount_sum,h15s_volume_sum,h15s_average_avg
0,2022-03-11 15:29:54,265.1000,265.1000,265.0967,265.1000,0.00,265.100,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-03-11 15:29:55,265.0867,265.1200,265.0867,265.1100,12.00,265.101,4,2022-03-11,3181.21200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2022-03-11 15:29:56,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2022-03-11 15:29:57,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2022-03-11 15:29:58,265.0900,265.0900,265.0533,265.0533,12.36,265.069,4,2022-03-11,3276.25284,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3612595,2022-07-25 12:29:49,270.1367,270.1367,270.1333,270.1367,4.68,270.135,1,2022-07-25,1264.23180,...,270.0833,5.0,51.00,13777.7550,270.152,270.2133,270.0833,5.0,51.00,270.152
3612596,2022-07-25 12:29:50,270.1367,270.1367,270.1333,270.1367,0.00,270.137,0,2022-07-25,0.00000,...,270.0833,5.0,52.68,14231.6958,270.154,270.2133,270.0833,5.0,52.68,270.154
3612597,2022-07-25 12:29:51,270.1700,270.1700,270.1700,270.1700,3.00,270.170,1,2022-07-25,810.51000,...,270.0833,5.0,52.68,14231.6958,270.154,270.2133,270.0833,5.0,52.68,270.154
3612598,2022-07-25 12:29:52,270.2100,270.2100,270.1533,270.1533,30.00,270.164,3,2022-07-25,8104.92000,...,270.0833,6.0,55.68,15042.2058,270.155,270.2133,270.0833,6.0,55.68,270.155


In [182]:
df = df.drop(columns=['_weighted_vol_avg'])

In [183]:
df.set_index('date')
df = df.sort_index(ascending=False)
df.reset_index()
df.head(100)

,date,open,high,low,close,volume,average,barCount,date2_str,h1s_high_max,...,h10s_barCount_sum,h10s_volume_sum,_h10s_weighted_vol_avg_sum,h10s_average_avg,h15s_high_max,h15s_low_min,h15s_barCount_sum,h15s_volume_sum,_h15s_weighted_vol_avg_sum,h15s_average_avg
3612599,2022-07-25 12:29:53,270.2100,270.2167,270.2100,270.2167,18.00,270.216,4,2022-07-25,270.2100,...,7.0,73.68,19904.66580,270.150,270.2100,270.0833,7.0,73.68,19904.66580,270.150
3612598,2022-07-25 12:29:52,270.2100,270.2100,270.1533,270.1533,30.00,270.164,3,2022-07-25,270.1700,...,6.0,55.68,15042.20580,270.155,270.2133,270.0833,6.0,55.68,15042.20580,270.155
3612597,2022-07-25 12:29:51,270.1700,270.1700,270.1700,270.1700,3.00,270.170,1,2022-07-25,270.1367,...,5.0,52.68,14231.69580,270.154,270.2133,270.0833,5.0,52.68,14231.69580,270.154
3612596,2022-07-25 12:29:50,270.1367,270.1367,270.1333,270.1367,0.00,270.137,0,2022-07-25,270.1367,...,5.0,52.68,14231.69580,270.154,270.2133,270.0833,5.0,52.68,14231.69580,270.154
3612595,2022-07-25 12:29:49,270.1367,270.1367,270.1333,270.1367,4.68,270.135,1,2022-07-25,270.1500,...,5.0,51.00,13777.75500,270.152,270.2133,270.0833,5.0,51.00,13777.75500,270.152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3612504,2022-07-25 12:28:18,269.9833,269.9833,269.9833,269.9833,3.00,269.983,1,2022-07-25,270.0100,...,10.0,54.51,14715.19578,269.954,270.0467,269.9067,10.0,54.51,14715.19578,269.954
3612503,2022-07-25 12:28:17,270.0100,270.0100,270.0067,270.0100,0.00,270.010,0,2022-07-25,270.0100,...,11.0,57.51,15525.32778,269.959,270.0467,269.9067,11.0,57.51,15525.32778,269.959
3612502,2022-07-25 12:28:16,270.0100,270.0100,270.0067,270.0100,0.00,270.010,0,2022-07-25,270.0100,...,11.0,57.51,15525.32778,269.959,270.0467,269.9067,11.0,57.51,15525.32778,269.959
3612501,2022-07-25 12:28:15,269.9433,270.0100,269.9400,270.0100,26.85,269.950,2,2022-07-25,270.0067,...,11.0,36.66,9897.63228,269.985,270.1133,269.9067,11.0,36.66,9897.63228,269.985


## Compute Future 5 second window summary

In [184]:
df['f5s_average'] = df['average'].rolling(window=6).agg( {'firstValue': firstValue})
df.head(100)

,date,open,high,low,close,volume,average,barCount,date2_str,h1s_high_max,...,h10s_volume_sum,_h10s_weighted_vol_avg_sum,h10s_average_avg,h15s_high_max,h15s_low_min,h15s_barCount_sum,h15s_volume_sum,_h15s_weighted_vol_avg_sum,h15s_average_avg,f5s_average
3612599,2022-07-25 12:29:53,270.2100,270.2167,270.2100,270.2167,18.00,270.216,4,2022-07-25,270.2100,...,73.68,19904.66580,270.150,270.2100,270.0833,7.0,73.68,19904.66580,270.150,NaN
3612598,2022-07-25 12:29:52,270.2100,270.2100,270.1533,270.1533,30.00,270.164,3,2022-07-25,270.1700,...,55.68,15042.20580,270.155,270.2133,270.0833,6.0,55.68,15042.20580,270.155,NaN
3612597,2022-07-25 12:29:51,270.1700,270.1700,270.1700,270.1700,3.00,270.170,1,2022-07-25,270.1367,...,52.68,14231.69580,270.154,270.2133,270.0833,5.0,52.68,14231.69580,270.154,NaN
3612596,2022-07-25 12:29:50,270.1367,270.1367,270.1333,270.1367,0.00,270.137,0,2022-07-25,270.1367,...,52.68,14231.69580,270.154,270.2133,270.0833,5.0,52.68,14231.69580,270.154,NaN
3612595,2022-07-25 12:29:49,270.1367,270.1367,270.1333,270.1367,4.68,270.135,1,2022-07-25,270.1500,...,51.00,13777.75500,270.152,270.2133,270.0833,5.0,51.00,13777.75500,270.152,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3612504,2022-07-25 12:28:18,269.9833,269.9833,269.9833,269.9833,3.00,269.983,1,2022-07-25,270.0100,...,54.51,14715.19578,269.954,270.0467,269.9067,10.0,54.51,14715.19578,269.954,269.977
3612503,2022-07-25 12:28:17,270.0100,270.0100,270.0067,270.0100,0.00,270.010,0,2022-07-25,270.0100,...,57.51,15525.32778,269.959,270.0467,269.9067,11.0,57.51,15525.32778,269.959,269.941
3612502,2022-07-25 12:28:16,270.0100,270.0100,270.0067,270.0100,0.00,270.010,0,2022-07-25,270.0100,...,57.51,15525.32778,269.959,270.0467,269.9067,11.0,57.51,15525.32778,269.959,269.910
3612501,2022-07-25 12:28:15,269.9433,270.0100,269.9400,270.0100,26.85,269.950,2,2022-07-25,270.0067,...,36.66,9897.63228,269.985,270.1133,269.9067,11.0,36.66,9897.63228,269.985,269.964


In [185]:
df = df.sort_index(ascending=True)
df.reset_index()
df.head(100)

,date,open,high,low,close,volume,average,barCount,date2_str,h1s_high_max,...,h10s_volume_sum,_h10s_weighted_vol_avg_sum,h10s_average_avg,h15s_high_max,h15s_low_min,h15s_barCount_sum,h15s_volume_sum,_h15s_weighted_vol_avg_sum,h15s_average_avg,f5s_average
0,2022-03-11 15:29:54,265.1000,265.1000,265.0967,265.1000,0.00,265.100,0,2022-03-11,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,265.053
1,2022-03-11 15:29:55,265.0867,265.1200,265.0867,265.1100,12.00,265.101,4,2022-03-11,265.1000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,265.001
2,2022-03-11 15:29:56,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,265.1200,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,264.952
3,2022-03-11 15:29:57,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,265.1100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,264.950
4,2022-03-11 15:29:58,265.0900,265.0900,265.0533,265.0533,12.36,265.069,4,2022-03-11,265.1100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,264.976
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2022-03-11 15:31:29,264.6867,264.6867,264.6833,264.6867,5.82,264.685,1,2022-03-11,264.7600,...,55.59,14714.04090,264.689,264.7667,264.6533,12.0,55.59,14714.04090,264.689,264.845
96,2022-03-11 15:31:30,264.7600,264.7600,264.7533,264.7533,24.12,264.759,4,2022-03-11,264.6867,...,58.11,15380.77650,264.684,264.7600,264.6533,12.0,58.11,15380.77650,264.684,264.900
97,2022-03-11 15:31:31,264.7533,264.7533,264.7533,264.7533,0.00,264.753,0,2022-03-11,264.7600,...,67.74,17931.81120,264.715,264.7600,264.6600,13.0,67.74,17931.81120,264.715,264.900
98,2022-03-11 15:31:32,264.8333,264.8333,264.6733,264.6733,9.00,264.780,2,2022-03-11,264.7533,...,58.77,15557.73924,264.722,264.7600,264.6600,12.0,58.77,15557.73924,264.722,264.900


In [186]:
%%time
pd.set_option('display.max_rows', 100)

df['f5s_10c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.10), axis=1)
df['f5s_15c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.15), axis=1)
df['f5s_20c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.20), axis=1)
df['f5s_25c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.25), axis=1)
df['f5s_30c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.30), axis=1)


In [187]:
df.head(100)

,date,open,high,low,close,volume,average,barCount,date2_str,h1s_high_max,...,h15s_barCount_sum,h15s_volume_sum,_h15s_weighted_vol_avg_sum,h15s_average_avg,f5s_average,f5s_10c_arrow,f5s_15c_arrow,f5s_20c_arrow,f5s_25c_arrow,f5s_30c_arrow
0,2022-03-11 15:29:54,265.1000,265.1000,265.0967,265.1000,0.00,265.100,0,2022-03-11,NaN,...,NaN,NaN,NaN,NaN,265.053,0,0,0,0,0
1,2022-03-11 15:29:55,265.0867,265.1200,265.0867,265.1100,12.00,265.101,4,2022-03-11,265.1000,...,NaN,NaN,NaN,NaN,265.001,-1,0,0,0,0
2,2022-03-11 15:29:56,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,265.1200,...,NaN,NaN,NaN,NaN,264.952,-1,-1,0,0,0
3,2022-03-11 15:29:57,265.1100,265.1100,265.1100,265.1100,0.00,265.110,0,2022-03-11,265.1100,...,NaN,NaN,NaN,NaN,264.950,-1,-1,0,0,0
4,2022-03-11 15:29:58,265.0900,265.0900,265.0533,265.0533,12.36,265.069,4,2022-03-11,265.1100,...,NaN,NaN,NaN,NaN,264.976,0,0,0,0,0
5,2022-03-11 15:29:59,265.0533,265.0533,265.0533,265.0533,0.00,265.053,0,2022-03-11,265.0900,...,NaN,NaN,NaN,NaN,265.020,0,0,0,0,0
6,2022-03-11 15:30:00,265.0033,265.0300,264.9767,265.0300,335.13,265.001,65,2022-03-11,265.0533,...,NaN,NaN,NaN,NaN,265.020,0,0,0,0,0
7,2022-03-11 15:30:01,264.9467,264.9767,264.9333,264.9500,24.15,264.952,5,2022-03-11,265.0300,...,NaN,NaN,NaN,NaN,265.020,0,0,0,0,0
8,2022-03-11 15:30:02,264.9500,264.9500,264.9500,264.9500,0.00,264.950,0,2022-03-11,264.9767,...,NaN,NaN,NaN,NaN,264.977,0,0,0,0,0
9,2022-03-11 15:30:03,264.9433,265.0000,264.9433,265.0000,33.00,264.976,8,2022-03-11,264.9500,...,NaN,NaN,NaN,NaN,265.070,0,0,0,0,0


In [188]:
%%time
df.to_csv("./TSLA-Stocks/ALL.csv", index=False)